# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriyaHarshini19/PixelToPrediction/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [ ]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [ ]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [ ]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [ ]:
# Your experiment here
# Experiment: Compare a deeper tree (max_depth=3)

tree3 = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

tree3.fit(X, y)

# Show learned rules
print(export_text(tree3, feature_names=features))

# Evaluate Precision@20 and Precision@50
tree3_score = tree3.predict_proba(X)[:, 1]

for k in (20, 50):
    print(f"Precision@{k}: {precision_at_k(tree3_score, y, k):.3f}")


Experiment: Testing the impact of increasing decision tree depth

Objective:
I increased the decision tree depth from 2 to 3 to observe whether allowing the model to learn more decision rules improves its ability to identify declining pages.

Method:
A new decision tree classifier was trained using the same observable features: content age, update frequency, impressions, average position, CTR, and word count. The model predictions were evaluated using Precision@20 and Precision@50.

Observation:
The depth-3 tree created more detailed decision paths compared to the depth-2 tree. This allowed the model to capture additional patterns in the data, but it also made the rules slightly more complex to interpret.

Learning:
Increasing model complexity does not always guarantee better performance. A useful ML model should maintain a balance between prediction quality and human interpretability.

Conclusion:
This experiment showed that deeper models can discover additional patterns, but simpler models may remain valuable because they are easier to understand, explain, and trust. The results are specific to this dataset and should be validated using unseen data before making final conclusions.


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.

In [ ]:
import numpy as np # Program 1: Bisection Method for finding roots print("PROGRAM 1: Bisection Method") def bisection_method(f, a, b, tol=1e-6, max_iter=100): """ Find root using Bisection Method """   if f(a) * f(b) > 0:         print("Root not bracketed in [a, b]")         return None      print(f"Iteration details:")     print(f"{'Iter':<6} {'a':<12} {'b':<12} {'c':<12} {'f(c)':<12}")     print("-"*55)      for i in range(max_iter):         c = (a + b) / 2         f_c = f(c)          print(f"{i+1:<6} {a:<12.6f} {b:<12.6f} {c:<12.6f} {f_c:<12.6f}")          if abs(f_c) < tol or abs(b - a) < tol:             return c          if f(a) * f_c < 0:             b = c         else:             a = c      return (a + b) / 2  # Define function f(x) = x^3 - x - 2 def f(x):     return x**3 - x - 2 print("Solving: x³ - x - 2 = 0") print("Initial interval: [1, 2]") root = bisection_method(f, 1, 2) if root is not None: print(f"\nRoot found: x = {root:.8f}") print(f"f({root:.8f}) = {f(root):.8f}") print("\nVerification with numpy:") print(f"All roots: {np.roots([1, 0, -1, -2])}")

In [ ]:
import numpy as np #Program 2: Bisection Method with User Input print("PROGRAM: Bisection Method")  def bisection(f, a, b, tol, max_iter):     if f(a)*f(b) > 0:         print("Root not bracketed!")         return None, 0      print(f"\n{'Iter':<6} {'a':<12} {'b':<12} {'c':<12} {'f(c)':<12}")     print("-"*60)      for i in range(max_iter):         c = (a + b) / 2         f_c = f(c)          print(f"{i+1:<6} {a:<12.6f} {b:<12.6f} {c:<12.6f} {f_c:<12.6f}")          if abs(f_c) < tol or abs(b-a) < tol:             return c, i+1          if f(a)*f_c < 0:             b = c         else:             a = c return (a+b)/2, max_iter # User Input print("\nEnter function in terms of x") print("Example: x**3 - x - 2") func_str = input("Enter f(x): ") # Check if input is empty if not func_str: func_str = "x**3 - x - 2" print(f"Using default: {func_str}") f = lambda x: eval(func_str) a = float(input("Enter lower limit (a): ")) b = float(input("Enter upper limit (b): ")) tol = float(input("Enter tolerance (e.g., 1e-6): ")) max_iter = int(input("Enter max iterations: ")) print(f"\nSolving: {func_str} = 0") print(f"Interval: [{a}, {b}]") root, iterations = bisection(f, a, b, tol, max_iter) if root is not None: print(f"\nRoot found: x = {root:.8f}") print(f"Iterations: {iterations}") print(f"Verification: f({root:.8f}) = {f(root):.8f}") else: print("\n  Failed to find root!") print("   Possible reasons:") print("   - f(a) and f(b) have same sign") print("   - Function is constant (like f(x)=5)") print("   - No root in the interval")

In [ ]:
import numpy as np # Program 3: Bisection Method - Multiple Equation print("PROGRAM 3: Bisection Method - Multiple Equations")  def bisection_general(f, a, b, tol=1e-6, max_iter=100):     if f(a) * f(b) > 0:         print(f"Root not bracketed in [{a}, {b}]")         return None      for i in range(max_iter):         c = (a + b) / 2         f_c = f(c)          if abs(f_c) < tol or abs(b - a) < tol:             return c          if f(a) * f_c < 0:             b = c         else:             a = c      return (a + b) / 2  # Test with different functions print("1. f(x) = x² - 4 (Roots: ±2)") def f1(x): return x**2 - 4 root1 = bisection_general(f1, 1, 3) print(f"Root in [1, 3]: x = {root1:.8f}\n") print("2. f(x) = sin(x) - 0.5 (Root: π/6)") def f2(x): return np.sin(x) - 0.5 root2 = bisection_general(f2, 0, 1) print(f"Root in [0, 1]: x = {root2:.8f}") print(f"sin({root2:.8f}) = {np.sin(root2):.8f}\n") print("3. f(x) = e^x - 5 (Root: ln(5))") def f3(x): return np.exp(x) - 5 root3 = bisection_general(f3, 1, 2) print(f"Root in [1, 2]: x = {root3:.8f}") print(f"e^{root3:.8f} = {np.exp(root3):.8f}")

In [ ]:
import numpy as np # Program 4: Newton-Raphson Method print("PROGRAM 4: Newton-Raphson Method")  def newton_raphson(f, df, x0, tol=1e-6, max_iter=100):     """     Find root using Newton-Raphson Method     """     print(f"Iteration details:")     print(f"{'Iter':<6} {'x':<14} {'f(x)':<14} {'Error':<14}")     print("-"*50)      x = x0     for i in range(max_iter):         fx = f(x)         dfx = df(x)          if abs(dfx) < 1e-12:             print("Derivative too small")             return None          x_new = x - fx / dfx         error = abs(x_new - x)          print(f"{i+1:<6} {x:<14.8f} {fx:<14.8f} {error:<14.8f}") if error < tol:             return x_new          x = x_new      return x  # Define f(x) = x^3 - x - 2 def f(x):     return x**3 - x - 2  def df(x):     return 3*x**2 - 1  print("Solving: x³ - x - 2 = 0") print(f"Initial guess: x0 = 1.5")  root = newton_raphson(f, df, 1.5)  if root is not None:     print(f"\nRoot found: x = {root:.10f}")     print(f"f({root:.10f}) = {f(root):.10f}")

In [ ]:
import numpy as np # Program 5: Newton-Raphson for Different Equations print("PROGRAM 5: Newton-Raphson - Multiple Equations")  def newton_raphson_general(f, df, x0, tol=1e-6, max_iter=100):     x = x0     for i in range(max_iter):         fx = f(x)         dfx = df(x)          if abs(dfx) < 1e-12:             return None          x_new = x - fx / dfx          if abs(x_new - x) < tol:             return x_new          x = x_new      return x  print("1. f(x) = cos(x) - x") def f1(x):     return np.cos(x) - x def df1(x): return -np.sin(x) - 1 root1 = newton_raphson_general(f1, df1, 0.5) print(f"Root: x = {root1:.10f}") print(f"cos({root1:.10f}) - {root1:.10f} = {f1(root1):.10f}\n") print("2. f(x) = x³ - 2x - 5") def f2(x): return x**3 - 2*x - 5 def df2(x): return 3*x**2 - 2 root2 = newton_raphson_general(f2, df2, 2) print(f"Root: x = {root2:.10f}") print(f"f({root2:.10f}) = {f2(root2):.10f}\n") print("3. f(x) = x - e^(-x)") def f3(x): return x - np.exp(-x) def df3(x): return 1 + np.exp(-x) root3 = newton_raphson_general(f3, df3, 0.5) print(f"Root: x = {root3:.10f}") print(f"{root3:.10f} - e^(-{root3:.10f}) = {f3(root3):.10f}")

In [ ]:
import numpy as np #Program 6: Newton-Raphson Method with user input print("="*50) print("PROGRAM-6: Newton-Raphson Method") print("="*50)  def newton_raphson(f, df, x0, tol, max_iter):     print(f"\n{'Iter':<6} {'x':<12} {'f(x)':<12} {'Error':<12}")     print("-"*45)      x = x0     for i in range(max_iter):         fx = f(x)         dfx = df(x)          if abs(dfx) < 1e-12:             print("Derivative is zero!")             return None, i          x_new = x - fx/dfx         error = abs(x_new - x)          print(f"{i+1:<6} {x:<12.6f} {fx:<12.6f} {error:<12.6f}")          if error < tol:             return x_new, i+1 x = x_new return x, max_iter # User Input print("\nEnter function in terms of x") print("Example: x**3 - x - 2") func_str = input("Enter f(x): ") f = lambda x: eval(func_str) print("\nEnter derivative in terms of x") print("Example: 3*x**2 - 1") deriv_str = input("Enter f'(x): ") df = lambda x: eval(deriv_str) x0 = float(input("Enter initial guess (x0): ")) tol = float(input("Enter tolerance (e.g., 1e-6): ")) max_iter = int(input("Enter max iterations: ")) print(f"\nSolving: {func_str} = 0") print(f"Initial guess: {x0}") root, iterations = newton_raphson(f, df, x0, tol, max_iter) if root is not None: print(f"\nRoot found: x = {root:.8f}") print(f"Iterations: {iterations}") print(f"Verification: f({root:.8f}) = {f(root):.8f}")